In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from joblib import Parallel, delayed
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
import scipy.stats as stats
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, LabelEncoder,PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [2]:
df=pd.read_csv('agronimy mandi dataset.csv')
des_df=pd.read_csv('des dataset.csv')
apy=pd.read_csv('APY.csv')
apy_2024_demo=pd.read_csv('2024_demo.csv')
apy_2025_demo=pd.read_csv('2025_demo.csv')

In [3]:
series = df[
    (df["State"].str.strip().str.lower() == "west bengal") &
    (df["District Name"].str.strip().str.lower() == "bankura") &
    (df["Market Name"].str.strip().str.lower() == "bishnupur(bankura)") &
    (df["Commodity"].str.strip().str.lower() == "wheat")
].copy()

print("Rows:", len(series))
print("Unique dates:", series["Price Date"].nunique())
print("Date range:", series["Price Date"].min(), "to", series["Price Date"].max())

print("\nDuplicate dates:")
print(series["Price Date"].duplicated().sum())

print("\nMissing dates:")
all_dates = pd.date_range(
    series["Price Date"].min(),
    series["Price Date"].max(),
    freq="D"
)

missing_dates = all_dates.difference(series["Price Date"])

print("Number of missing calendar dates:", len(missing_dates))
print("First missing dates:", missing_dates[:20].tolist())

Rows: 364
Unique dates: 364
Date range: 01 Apr 2025 to 31 Oct 2024

Duplicate dates:
0

Missing dates:
Number of missing calendar dates: 0
First missing dates: []


In [4]:
master_df = series[
    [
        "State",
        "District Name",
        "Market Name",
        "Commodity",
        "Price Date",
        "Min Price (Rs./Quintal)",
        "Max Price (Rs./Quintal)",
        "Modal Price (Rs./Quintal)"
    ]
].copy()

master_df = master_df.sort_values("Price Date").reset_index(drop=True)

print("Shape:", master_df.shape)
display(master_df.head())
display(master_df.tail())

Shape: (364, 8)


,State,District Name,Market Name,Commodity,Price Date,Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal)
0,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Apr 2025,2800.0,3100.0,2900.0
1,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Aug 2025,2800.0,3200.0,3000.0
2,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Dec 2024,2600.0,2800.0,2700.0
3,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Feb 2025,2700.0,3000.0,2800.0
4,West Bengal,Bankura,Bishnupur(Bankura),Wheat,01 Jan 2025,2750.0,3000.0,2850.0


,State,District Name,Market Name,Commodity,Price Date,Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal)
359,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 Jan 2025,2700.0,3000.0,2800.0
360,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 Jul 2025,2800.0,3200.0,3000.0
361,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 Mar 2025,2800.0,3100.0,2900.0
362,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 May 2025,2800.0,3200.0,3000.0
363,West Bengal,Bankura,Bishnupur(Bankura),Wheat,31 Oct 2024,2700.0,3000.0,2800.0


In [6]:
master_df["Price Date"] = pd.to_datetime(
    master_df["Price Date"],
    format="%d %b %Y",
    errors="coerce"
)

master_df = master_df.sort_values("Price Date").reset_index(drop=True)

print("Date dtype:", master_df["Price Date"].dtype)
print("Start:", master_df["Price Date"].min())
print("End:", master_df["Price Date"].max())



Date dtype: datetime64[us]
Start: 2024-08-15 00:00:00
End: 2025-08-14 00:00:00


In [7]:
master_df["Year"] = master_df["Price Date"].dt.year
master_df["Month"] = master_df["Price Date"].dt.month
master_df["Month_Name"] = master_df["Price Date"].dt.month_name()
master_df["Quarter"] = master_df["Price Date"].dt.quarter
master_df["Day_of_Week"] = master_df["Price Date"].dt.dayofweek
master_df["Day_Name"] = master_df["Price Date"].dt.day_name()
master_df["Day_of_Year"] = master_df["Price Date"].dt.dayofyear

print(master_df[
    [
        "Price Date",
        "Year",
        "Month",
        "Month_Name",
        "Quarter",
        "Day_of_Week",
        "Day_Name",
        "Day_of_Year"
    ]
].head(10))

  Price Date  Year  Month Month_Name  Quarter  Day_of_Week   Day_Name  \
0 2024-08-15  2024      8     August        3            3   Thursday   
1 2024-08-16  2024      8     August        3            4     Friday   
2 2024-08-17  2024      8     August        3            5   Saturday   
3 2024-08-18  2024      8     August        3            6     Sunday   
4 2024-08-19  2024      8     August        3            0     Monday   
5 2024-08-20  2024      8     August        3            1    Tuesday   
6 2024-08-21  2024      8     August        3            2  Wednesday   
7 2024-08-22  2024      8     August        3            3   Thursday   
8 2024-08-23  2024      8     August        3            4     Friday   
9 2024-08-24  2024      8     August        3            5   Saturday   

   Day_of_Year  
0          228  
1          229  
2          230  
3          231  
4          232  
5          233  
6          234  
7          235  
8          236  
9          237  


In [ ]:
def get_price_season(month):
    if month in [6, 7, 8, 9, 10]:
        return "Kharif"
    elif month in [11, 12, 1, 2]:
        return "Rabi"
    else:
        return "Summer"


master_df["Price_Season"] = master_df["Month"].apply(get_price_season)

# Wheat's agricultural season according to our APY data
master_df["Crop_Season"] = "Rabi"

print(
    master_df[
        ["Price Date", "Year", "Month", "Price_Season", "Crop_Season"]
    ].head(20)
)

print("\nPrice season distribution:")
print(master_df["Price_Season"].value_counts())

In [ ]:
apy_24.columns = apy_24.columns.str.strip()
apy_25.columns = apy_25.columns.str.strip()

print(apy_24.columns.tolist())
print(apy_25.columns.tolist())